# ML & Sensing Final Project: Breathing Pattern Detection

**Authors**: Maanvi Sarwadi, Alina Zacaria

In [16]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import glob


from google.colab import drive
from scipy.fft import fft, fftfreq
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [17]:
try:
    drive.mount('/content/drive', force_remount=True)
    print("Drive mounted.")
except:
    print("Error mounting Drive, try running this cell again.")

Mounted at /content/drive
Drive mounted.


In [18]:
# Load dataset
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("arashnic/lung-dataset")

# print("Path to dataset files:", path)

In [19]:
def extract_audio_features(file_path):
    audio, sr = librosa.load(file_path, sr=22050)

    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
    zero_crossing_rate = librosa.feature.zero_crossing_rate(audio)

    features = np.hstack([
        np.mean(mfccs, axis=1),
        np.mean(spectral_centroid),
        np.mean(zero_crossing_rate)
    ])

    return features

In [20]:
# Column names for PhyPhox accelerometer CSV
# Adjust if your app exports different column names
ACCEL_COLS = {
    'time': 'time (s)',
    'x':    'Linear Acceleration x (m/s^2)',
    'y':    'Linear Acceleration y (m/s^2)',
    'z':    'Linear Acceleration z (m/s^2)',
}

PASSIVE_AUDIO_DIR = os.path.join(DATA_ROOT, "passive_audio")
ACTIVE_AUDIO_DIR = os.path.join(DATA_ROOT, "active_audio")
PASSIVE_ACCEL_DIR = os.path.join(DATA_ROOT, "passive_accelerator")

LABELS = {'normal': 0, 'irregular': 1}
CLASS_NAMES = ['Normal', 'Irregular']

DATA_ROOT   = "/content/drive/MyDrive/MLaS-final-ds"         # root folder (see directory structure above)
AUDIO_SR    = 22050                                          # audio sample rate (Hz)
N_MFCC      = 13                                             # number of MFCC coefficients
ACCEL_SR    = 100                                            # accelerometer sample rate (Hz); PhyPhox default
BREATH_LOW  = 0.1                                            # breathing band lower bound (Hz) — ~6 breaths/min
BREATH_HIGH = 0.5                                            # breathing band upper bound (Hz) — ~30 breaths/min

# Matching feature names (used for importance plots later)
AUDIO_FEATURE_NAMES = (
    [f'mfcc_{i}_mean' for i in range(N_MFCC)] +
    [f'mfcc_{i}_std'  for i in range(N_MFCC)] +
    ['centroid_mean', 'centroid_std',
     'zcr_mean',      'zcr_std',
     'rms_mean',      'rms_std',
     'rolloff_mean',  'rolloff_std']
)

print(f"Audio feature vector length: {len(AUDIO_FEATURE_NAMES)}")


Audio feature vector length: 34


In [21]:
def load_passive_accel_files(accel_root):
    csv_files = []
    for folder in sorted(os.listdir(accel_root)):
        folder_path = os.path.join(accel_root, folder)
        if os.path.isdir(folder_path):
            csv_path = os.path.join(folder_path, "Accelerometer.csv")
            if os.path.exists(csv_path):
                csv_files.append(csv_path)
    return csv_files


def extract_accel_features(file_path, accel_sr=ACCEL_SR,
                           breath_low=BREATH_LOW, breath_high=BREATH_HIGH):
    """
    Returns a 1D feature vector from an accelerometer CSV.
    Total features: 15
    """
    df = load_passive_accel_files(file_path)

    # ── Time-domain features ─────────────────────────────────────────────────
    magnitude = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2).values

    time_features = np.array([
        np.mean(magnitude),
        np.std(magnitude),
        np.max(magnitude),
        np.min(magnitude),
        np.max(magnitude) - np.min(magnitude),  # range
        np.var(magnitude),
        np.mean(df['x'].values), np.std(df['x'].values),
        np.mean(df['y'].values), np.std(df['y'].values),
        np.mean(df['z'].values), np.std(df['z'].values),
    ])

    # ── Frequency-domain features (FFT) ──────────────────────────────────────
    n       = len(magnitude)
    freqs   = fftfreq(n, d=1.0 / accel_sr)
    fft_mag = np.abs(fft(magnitude))

    # Keep only positive frequencies
    pos_mask   = freqs > 0
    freqs_pos  = freqs[pos_mask]
    fft_pos    = fft_mag[pos_mask]

    # Breathing band (0.1 – 0.5 Hz)
    breath_mask    = (freqs_pos >= breath_low) & (freqs_pos <= breath_high)
    breath_power   = np.sum(fft_pos[breath_mask]) if breath_mask.sum() > 0 else 0.0
    dominant_freq  = (
        freqs_pos[breath_mask][np.argmax(fft_pos[breath_mask])]
        if breath_mask.sum() > 0 else 0.0
    )
    total_power    = np.sum(fft_pos) + 1e-8
    breath_ratio   = breath_power / total_power

    freq_features = np.array([dominant_freq, breath_power, breath_ratio])

    return np.hstack([time_features, freq_features])


ACCEL_FEATURE_NAMES = [
    'mag_mean', 'mag_std', 'mag_max', 'mag_min', 'mag_range', 'mag_var',
    'x_mean', 'x_std', 'y_mean', 'y_std', 'z_mean', 'z_std',
    'dominant_freq_hz', 'breath_band_power', 'breath_power_ratio',
]

print(f"Accelerometer feature vector length: {len(ACCEL_FEATURE_NAMES)}")

Accelerometer feature vector length: 15


In [25]:
ALL_FEATURE_NAMES = AUDIO_FEATURE_NAMES + ACCEL_FEATURE_NAMES
print(f"Total fused feature vector length: {len(ALL_FEATURE_NAMES)}")


def build_dataset(data_root=DATA_ROOT):
    """
    Walks data_root/<label>/audio/ and data_root/<label>/accel/,
    pairs files by sorted name, extracts features, and returns X, y.
    """

    passive_audio_files = sorted(glob.glob(os.path.join(PASSIVE_AUDIO_DIR, "*.wav")))
    active_audio_files = sorted(glob.glob(os.path.join(ACTIVE_AUDIO_DIR, "*.wav")))
    passive_accel_files = sorted(glob.glob(os.path.join(PASSIVE_ACCEL_DIR, "*", "Accelerometer.csv")))

    X_rows, y_rows, metarows = [], [], []

    passive_n = min(len(passive_audio_files), len(passive_accel_files))

    for i in range(passive_n):
        audio_path = passive_audio_files[i]
        accel_path = passive_accel_files[i]

        try:
            audio_feat = extract_audio_features(audio_path)
            df = pd.read_csv(accel_path)
            df = df.rename(columns={v: k for k, v in ACCEL_COLS.items()})
            magnitude = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2).values
            
            n = len(magnitude)
            freqs = fftfreq(n, d=1.0 / ACCEL_SR)
            fft_mag = np.abs(fft(magnitude))
            pos_mask = freqs > 0
            freqs_pos, fft_pos = freqs[pos_mask], fft_mag[pos_mask]
            breath_mask = (freqs_pos >= BREATH_LOW) & (freqs_pos <= BREATH_HIGH)
            
            accel_feat = np.array([
                np.mean(magnitude), np.std(magnitude), np.max(magnitude), np.min(magnitude),
                np.max(magnitude) - np.min(magnitude), np.var(magnitude),
                np.mean(df['x']), np.std(df['x']), np.mean(df['y']), np.std(df['y']),
                np.mean(df['z']), np.std(df['z']),
                freqs_pos[breath_mask][np.argmax(fft_pos[breath_mask])] if breath_mask.sum() > 0 else 0.0,
                np.sum(fft_pos[breath_mask]) if breath_mask.sum() > 0 else 0.0,
                np.sum(fft_pos[breath_mask]) / (np.sum(fft_pos) + 1e-8)
            ])
            
            X_rows.append(np.concatenate([audio_feat, accel_feat]))
            y_rows.append(LABELS["normal"])
            metarows.append({"label": "normal", "audio_path": audio_path, "accel_path": accel_path})
        except Exception as e:
            print(f"[ERROR] Passive pair {i}: {e}")

    for audio_path in active_audio_files:
        try:
            audio_feat = extract_audio_features(audio_path)
            accel_feat = np.zeros(15, dtype=float)
            X_rows.append(np.concatenate([audio_feat, accel_feat]))
            y_rows.append(LABELS["irregular"])
            metarows.append({"label": "irregular", "audio_path": audio_path, "accel_path": None})
        except Exception as e:
            print(f"[ERROR] Active file {audio_path}: {e}")

    X = np.array(X_rows, dtype=float)
    y = np.array(y_rows, dtype=int)
    meta = pd.DataFrame(metarows)

    return X, y, meta


X, y, meta = build_dataset()
print("X shape:", X.shape)
print("y shape:", y.shape)
print(meta["label"].value_counts())

print(f"\nDataset shape : X = {X.shape}, y = {y.shape}")
print(f"Class balance : Resting = {np.sum(y == 0)}, Active = {np.sum(y == 1)}")

Total fused feature vector length: 49
X shape: (20, 30)
y shape: (20,)
label
normal       10
irregular    10
Name: count, dtype: int64

Dataset shape : X = (20, 30), y = (20,)
Class balance : Resting = 10, Active = 10


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
import pandas as pd

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0)
    })

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_base = baseline.predict(X_test)

results.append({
    "model": "Baseline (always normal)",
    "accuracy": accuracy_score(y_test, y_base),
    "precision": precision_score(y_test, y_base, zero_division=0),
    "recall": recall_score(y_test, y_base, zero_division=0),
    "f1": f1_score(y_test, y_base, zero_division=0)
})

results_df = pd.DataFrame(results)
print(results_df)

best_name = results_df.sort_values("f1", ascending=False).iloc[0]["model"]
if best_name in models:
    best_model = models[best_name]
    best_model.fit(X_train, y_train)
    y_best = best_model.predict(X_test)
else:
    y_best = y_base


                      model  accuracy  precision  recall   f1
0             Random Forest       1.0        1.0     1.0  1.0
1       Logistic Regression       1.0        1.0     1.0  1.0
2  Baseline (always normal)       0.5        0.0     0.0  0.0


In [ ]:
TODO 
'''
### Error Analysis ###
- confusion matrix
- Identify missed irregular breathing cases

### Real-Time Performance ###
- latency of predictions
- stability over continuous sensing

'''


In [ ]:
# Visualize accelerometer signal
# plt.figure(figsize=(12, 4))
# plt.plot(accel_df['x'], label='X-axis')
# plt.plot(accel_df['y'], label='Y-axis')
# plt.plot(accel_df['z'], label='Z-axis')
# plt.title("Accelerometer Signal")
# plt.xlabel("Time")
# plt.ylabel("Acceleration")
# plt.legend()
# plt.show()

In [ ]:
# Extract features from accelerometer data
# def extract_accelerometer_features(df):

#     magnitude = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

#     features = [
#         np.mean(magnitude),
#         np.std(magnitude),
#         np.max(magnitude),
#         np.min(magnitude),
#         np.var(magnitude)
#     ]

#     return np.array(features)